# 1_API — Data Preparation

**Author:** Sota Yoshimoto  
Builds the master panel (`output/df.csv`) used by `2_main.ipynb`.

**Steps**
1. Load local registration CSV/XLSX files
2. Fetch control variables from the SCB API
   (education, income, urbanization, households, density, electricity price)
3. Merge yearly stock and new-registration data, add `prev_` lag columns,
   and compute scrapping counts
4. Save long-format panel (290 kommuner × 2019–2024)

Raw inputs are expected under `./data/` and outputs are written to `./output/`.
See the repository README for data sources.


In [ ]:
from pathlib import Path

import requests
import pandas as pd
import statsmodels.api as sm

# ---------- Path configuration ----------
# Raw inputs live under DATA_DIR; processed outputs are written to OUTPUT_DIR.
# Adjust these two paths if your local layout differs.
DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)


## 1. Load local data

The multi-sheet registration Excel file and the owner-type Excel file
are read from `data/data_new/`.


### 1.1 Registration Excel (2018–2024)

`df_registration.xlsx` contains one sheet per year for the stock
(`2018`, `2019`, …) and one for new registrations (`2018_new`, …),
plus a `kommun_group` sheet with the SKR 9-group classification.


In [ ]:
# Read every sheet of the registration workbook and clean each one:
#   - strip column names
#   - replace SCB's missing-data marker (' –') with 0
#   - drop rows whose Kommun-kod / Kommunkod is missing or aggregate ("AA")
#   - left-pad the kommun code to 4 digits to preserve leading zeros
import pandas as pd

file_path = DATA_DIR / "data_new" / "df_registration.xlsx"

# Read all sheets
sheets = pd.read_excel(file_path, sheet_name=None, dtype={"Kommun-kod": str})
cleaned_sheets = {}

for sheet_name, df in sheets.items():
    # Work on a copy
    df = df.copy()

    # Clean column names
    df.columns = df.columns.str.strip()

    df = df.replace(" –", 0)

    if "Kommun-kod" in df.columns:
        df["Kommun-kod"] = (
            df["Kommun-kod"]
            .astype(str)
            .replace("nan", pd.NA)
        )
        df = df[~df["Kommun-kod"].str.contains("AA", na=False)]
        df = df.dropna(subset=["Kommun-kod"])
        df["Kommun-kod"] = df["Kommun-kod"].str.zfill(4)

    # Only process if Municipality column exists
    if "Municipality" in df.columns:
        # Ensure string type
        df["Municipality"] = df["Municipality"].astype(str)

        # Remove rows where Municipality is missing or empty
        df["Municipality"] = df["Municipality"].replace("nan", pd.NA)
        df = df.dropna(subset=["Municipality"])

    if "Kommunkod" in df.columns:
        df["Kommunkod"] = (
            df["Kommunkod"]
            .astype(str)
            .replace("nan", pd.NA)
        )
        df = df[~df["Kommunkod"].str.contains("AA", na=False)]
        df = df.dropna(subset=["Kommunkod"])
        df["Kommunkod"] = df["Kommunkod"].str.zfill(4)
    
    cleaned_sheets[sheet_name] = df


In [ ]:
# Attach the SKR 9-group classification (Gruppkod) to every yearly sheet,
# matching on the 4-digit Kommun-kod.
df_group = cleaned_sheets["kommun_group"].copy()

# Ensure municipality code is a 4-digit string
df_group["Kommunkod"] = (
    df_group["Kommunkod"]
    .astype(str)
    .replace("nan", "")
    .str.zfill(4)
)

# Rename key column to match other DataFrames
df_group = (
    df_group
    .rename(columns={"Kommunkod": "Kommun-kod"})
    [["Kommun-kod", "Gruppkod"]]
    .drop_duplicates()
)

# Merge Gruppkod into every yearly sheet except kommun_group itself
for sheet_name, df in cleaned_sheets.items():

    # Skip kommun_group itself
    if sheet_name == "kommun_group":
        continue

    df = df.copy()

    # Only sheets keyed by 'Kommun-kod' need merging
    if "Kommun-kod" in df.columns:
        df["Kommun-kod"] = (
            df["Kommun-kod"]
            .astype(str)
            .replace("nan", "")
            .str.zfill(4)
        )

        df = df.merge(df_group, on="Kommun-kod", how="left")
    # Update dictionary with merged DataFrame
    cleaned_sheets[sheet_name] = df


In [ ]:
# For each year, outer-join the stock sheet (e.g. "2019") with the
# corresponding new-registration sheet ("2019_new") on Kommun-kod.
# Suffix collisions are resolved by keeping the stock column and dropping
# any duplicated column ending with `_drop`.
merged_dfs = {}
years = [2018, 2019, 2020, 2021, 2022, 2023, 2024]

for year in years:
    sheet_main = str(year)
    sheet_new = f"{year}_new"

    if sheet_main in cleaned_sheets and sheet_new in cleaned_sheets:
        df_stock = cleaned_sheets[sheet_main]
        df_new = cleaned_sheets[sheet_new]

        df_merged = pd.merge(
            df_stock,
            df_new,
            on="Kommun-kod",
            how="outer",  
            suffixes=("", "_drop")
        )
        
        df_merged = df_merged[[c for c in df_merged.columns if not c.endswith("_drop")]]
        
        merged_dfs[year] = df_merged

df_2018 = merged_dfs.get(2018)
df_2019 = merged_dfs.get(2019)
df_2020 = merged_dfs.get(2020)
df_2021 = merged_dfs.get(2021)
df_2022 = merged_dfs.get(2022)
df_2023 = merged_dfs.get(2023)
df_2024 = merged_dfs.get(2024)


## 2. Fetch control variables from the SCB API

All API calls below post a query against SCB's PXWeb endpoint and unpack
the response into a tidy long-format DataFrame keyed by `Region` (4-digit
kommun-kod) and `Year`. The same list of 290 kommun codes is repeated in
each `payload["query"]["Region"]["values"]` block — kept verbatim for
transparency against the SCB API.


### 2.1 Education (2018–2024)

SCB table `UF0506B/Utbildning`. Returns population counts by
education level (ISCED 1–7 and US = unknown), sex, and year for the
16–74 age group. Bachelor's-or-higher share is computed as
(levels 6 + 7) / total.


In [ ]:
# Endpoint: SCB UF0506B/Utbildning — population by education level, sex,
# age group, and year. Returns one row per (Region, Age, Level, Sex, Year).
url = "https://api.scb.se/OV0104/v1/doris/en/ssd/START/UF/UF0506/UF0506B/Utbildning"
payload = {
  "query": [
    {
      "code": "Region",
      "selection": {
        "filter": "vs:RegionKommun07",
        "values": [
          "0114",
          "0115",
          "0117",
          "0120",
          "0123",
          "0125",
          "0126",
          "0127",
          "0128",
          "0136",
          "0138",
          "0139",
          "0140",
          "0160",
          "0162",
          "0163",
          "0180",
          "0181",
          "0182",
          "0183",
          "0184",
          "0186",
          "0187",
          "0188",
          "0191",
          "0192",
          "0305",
          "0319",
          "0330",
          "0331",
          "0360",
          "0380",
          "0381",
          "0382",
          "0428",
          "0461",
          "0480",
          "0481",
          "0482",
          "0483",
          "0484",
          "0486",
          "0488",
          "0509",
          "0512",
          "0513",
          "0560",
          "0561",
          "0562",
          "0563",
          "0580",
          "0581",
          "0582",
          "0583",
          "0584",
          "0586",
          "0604",
          "0617",
          "0642",
          "0643",
          "0662",
          "0665",
          "0680",
          "0682",
          "0683",
          "0684",
          "0685",
          "0686",
          "0687",
          "0760",
          "0761",
          "0763",
          "0764",
          "0765",
          "0767",
          "0780",
          "0781",
          "0821",
          "0834",
          "0840",
          "0860",
          "0861",
          "0862",
          "0880",
          "0881",
          "0882",
          "0883",
          "0884",
          "0885",
          "0980",
          "1060",
          "1080",
          "1081",
          "1082",
          "1083",
          "1214",
          "1230",
          "1231",
          "1233",
          "1256",
          "1257",
          "1260",
          "1261",
          "1262",
          "1263",
          "1264",
          "1265",
          "1266",
          "1267",
          "1270",
          "1272",
          "1273",
          "1275",
          "1276",
          "1277",
          "1278",
          "1280",
          "1281",
          "1282",
          "1283",
          "1284",
          "1285",
          "1286",
          "1287",
          "1290",
          "1291",
          "1292",
          "1293",
          "1315",
          "1380",
          "1381",
          "1382",
          "1383",
          "1384",
          "1401",
          "1402",
          "1407",
          "1415",
          "1419",
          "1421",
          "1427",
          "1430",
          "1435",
          "1438",
          "1439",
          "1440",
          "1441",
          "1442",
          "1443",
          "1444",
          "1445",
          "1446",
          "1447",
          "1452",
          "1460",
          "1461",
          "1462",
          "1463",
          "1465",
          "1466",
          "1470",
          "1471",
          "1472",
          "1473",
          "1480",
          "1481",
          "1482",
          "1484",
          "1485",
          "1486",
          "1487",
          "1488",
          "1489",
          "1490",
          "1491",
          "1492",
          "1493",
          "1494",
          "1495",
          "1496",
          "1497",
          "1498",
          "1499",
          "1715",
          "1730",
          "1737",
          "1760",
          "1761",
          "1762",
          "1763",
          "1764",
          "1765",
          "1766",
          "1780",
          "1781",
          "1782",
          "1783",
          "1784",
          "1785",
          "1814",
          "1860",
          "1861",
          "1862",
          "1863",
          "1864",
          "1880",
          "1881",
          "1882",
          "1883",
          "1884",
          "1885",
          "1904",
          "1907",
          "1960",
          "1961",
          "1962",
          "1980",
          "1981",
          "1982",
          "1983",
          "1984",
          "2021",
          "2023",
          "2026",
          "2029",
          "2031",
          "2034",
          "2039",
          "2061",
          "2062",
          "2080",
          "2081",
          "2082",
          "2083",
          "2084",
          "2085",
          "2101",
          "2104",
          "2121",
          "2132",
          "2161",
          "2180",
          "2181",
          "2182",
          "2183",
          "2184",
          "2260",
          "2262",
          "2280",
          "2281",
          "2282",
          "2283",
          "2284",
          "2303",
          "2305",
          "2309",
          "2313",
          "2321",
          "2326",
          "2361",
          "2380",
          "2401",
          "2403",
          "2404",
          "2409",
          "2417",
          "2418",
          "2421",
          "2422",
          "2425",
          "2460",
          "2462",
          "2463",
          "2480",
          "2481",
          "2482",
          "2505",
          "2506",
          "2510",
          "2513",
          "2514",
          "2518",
          "2521",
          "2523",
          "2560",
          "2580",
          "2581",
          "2582",
          "2583",
          "2584"
        ]
      }
    },
    {
      "code": "Alder",
      "selection": {
        "filter": "vs:ÅlderTotB",
        "values": [
          "tot16-74"
        ]
      }
    },
    {
      "code": "UtbildningsNiva",
      "selection": {
        "filter": "item",
        "values": [
          "1",
          "2",
          "3",
          "4",
          "5",
          "6",
          "7",
          "US"
        ]
      }
    },
    {
      "code": "Kon",
      "selection": {
        "filter": "item",
        "values": [
          "1",
          "2"
        ]
      }
    },
    {
      "code": "Tid",
      "selection": {
        "filter": "item",
        "values": [
          "2018",
          "2019",
          "2020",
          "2021",
          "2022",
          "2023",
          "2024"
        ]
      }
    }
  ],
  "response": {
    "format": "json"
  }
}
response = requests.post(url, json=payload)
response.raise_for_status()
data = response.json()


In [ ]:
# Unpack the API response into a long DataFrame. Each `data` entry contains
# a list of dimension keys (in column order) and a single measurement value.
rows = []

for item in data["data"]:
    row = {
        "Region": item["key"][0],
        "Age": item["key"][1],
        "EducationLevel": item["key"][2],
        "Sex": item["key"][3],
        "Year": item["key"][4],
        "Value": item["values"][0]
    }
    rows.append(row)

df_education = pd.DataFrame(rows)

# Convert "Value" column to numeric types (float/int)
# errors='coerce' transforms invalid parsing (like ".." or "-") into NaN
df_education["Value"] = pd.to_numeric(df_education["Value"], errors='coerce')
df_education["Year"] = df_education["Year"].astype(int)

In [ ]:
# Compute the share of the 16–74 population with a bachelor's degree or
# higher. Numerator = ISCED levels 6 and 7 (both sexes); denominator =
# all education levels (both sexes) in the same (Region, Year) cell.
# Denominator: total population (all education levels, both sexes)
total = (
    df_education
    .groupby(["Region", "Year"], as_index=False)["Value"]
    .sum()
    .rename(columns={"Value": "sum"})
)

# Numerator: population with bachelor's degree or higher
# EducationLevel '6' and '7', both sexes combined
bachelor_plus = (
    df_education[df_education["EducationLevel"].isin(["6", "7"])]
    .groupby(["Region", "Year"], as_index=False)["Value"]
    .sum()
    .rename(columns={"Value": "bachelor_plus_sum"})
)

# Merge numerator and denominator
df_education = total.merge(
    bachelor_plus,
    on=["Region", "Year"],
    how="left"
)

# Replace NaN with zero when bachelor-plus population is missing
df_education["bachelor_plus_sum"] = df_education["bachelor_plus_sum"].fillna(0)

# Compute share of population with bachelor's degree or higher
df_education["bachelor_plus_ratio"] = (
    df_education["bachelor_plus_sum"] / df_education["sum"]
)

# Percentage version (0–100 scale)
df_education["bachelor_plus_pct"] = df_education["bachelor_plus_ratio"] * 100


### 2.2 Median disposable income

SCB table `HE0110G/TabVX4bDispInkN`. Median disposable household income
(thousand SEK) for households of any composition (`E90`) where the
reference person is 18+. The growth rate is the per-region year-on-year
percentage change.


In [ ]:
# Endpoint: SCB HE0110G/TabVX4bDispInkN — median disposable income by
# household type, age, and year. Filtered to all households (E90) and
# reference person 18+, with ContentsCode 000006SY (median).
import requests
url = "https://api.scb.se/OV0104/v1/doris/en/ssd/START/HE/HE0110/HE0110G/TabVX4bDispInkN"
payload = {
  "query": [
    {
      "code": "Region",
      "selection": {
        "filter": "vs:RegionKommun07EjAggr",
        "values": [
          "0114",
          "0115",
          "0117",
          "0120",
          "0123",
          "0125",
          "0126",
          "0127",
          "0128",
          "0136",
          "0138",
          "0139",
          "0140",
          "0160",
          "0162",
          "0163",
          "0180",
          "0181",
          "0182",
          "0183",
          "0184",
          "0186",
          "0187",
          "0188",
          "0191",
          "0192",
          "0305",
          "0319",
          "0330",
          "0331",
          "0360",
          "0380",
          "0381",
          "0382",
          "0428",
          "0461",
          "0480",
          "0481",
          "0482",
          "0483",
          "0484",
          "0486",
          "0488",
          "0509",
          "0512",
          "0513",
          "0560",
          "0561",
          "0562",
          "0563",
          "0580",
          "0581",
          "0582",
          "0583",
          "0584",
          "0586",
          "0604",
          "0617",
          "0642",
          "0643",
          "0662",
          "0665",
          "0680",
          "0682",
          "0683",
          "0684",
          "0685",
          "0686",
          "0687",
          "0760",
          "0761",
          "0763",
          "0764",
          "0765",
          "0767",
          "0780",
          "0781",
          "0821",
          "0834",
          "0840",
          "0860",
          "0861",
          "0862",
          "0880",
          "0881",
          "0882",
          "0883",
          "0884",
          "0885",
          "0980",
          "1060",
          "1080",
          "1081",
          "1082",
          "1083",
          "1214",
          "1230",
          "1231",
          "1233",
          "1256",
          "1257",
          "1260",
          "1261",
          "1262",
          "1263",
          "1264",
          "1265",
          "1266",
          "1267",
          "1270",
          "1272",
          "1273",
          "1275",
          "1276",
          "1277",
          "1278",
          "1280",
          "1281",
          "1282",
          "1283",
          "1284",
          "1285",
          "1286",
          "1287",
          "1290",
          "1291",
          "1292",
          "1293",
          "1315",
          "1380",
          "1381",
          "1382",
          "1383",
          "1384",
          "1401",
          "1402",
          "1407",
          "1415",
          "1419",
          "1421",
          "1427",
          "1430",
          "1435",
          "1438",
          "1439",
          "1440",
          "1441",
          "1442",
          "1443",
          "1444",
          "1445",
          "1446",
          "1447",
          "1452",
          "1460",
          "1461",
          "1462",
          "1463",
          "1465",
          "1466",
          "1470",
          "1471",
          "1472",
          "1473",
          "1480",
          "1481",
          "1482",
          "1484",
          "1485",
          "1486",
          "1487",
          "1488",
          "1489",
          "1490",
          "1491",
          "1492",
          "1493",
          "1494",
          "1495",
          "1496",
          "1497",
          "1498",
          "1499",
          "1715",
          "1730",
          "1737",
          "1760",
          "1761",
          "1762",
          "1763",
          "1764",
          "1765",
          "1766",
          "1780",
          "1781",
          "1782",
          "1783",
          "1784",
          "1785",
          "1814",
          "1860",
          "1861",
          "1862",
          "1863",
          "1864",
          "1880",
          "1881",
          "1882",
          "1883",
          "1884",
          "1885",
          "1904",
          "1907",
          "1960",
          "1961",
          "1962",
          "1980",
          "1981",
          "1982",
          "1983",
          "1984",
          "2021",
          "2023",
          "2026",
          "2029",
          "2031",
          "2034",
          "2039",
          "2061",
          "2062",
          "2080",
          "2081",
          "2082",
          "2083",
          "2084",
          "2085",
          "2101",
          "2104",
          "2121",
          "2132",
          "2161",
          "2180",
          "2181",
          "2182",
          "2183",
          "2184",
          "2260",
          "2262",
          "2280",
          "2281",
          "2282",
          "2283",
          "2284",
          "2303",
          "2305",
          "2309",
          "2313",
          "2321",
          "2326",
          "2361",
          "2380",
          "2401",
          "2403",
          "2404",
          "2409",
          "2417",
          "2418",
          "2421",
          "2422",
          "2425",
          "2460",
          "2462",
          "2463",
          "2480",
          "2481",
          "2482",
          "2505",
          "2506",
          "2510",
          "2513",
          "2514",
          "2518",
          "2521",
          "2523",
          "2560",
          "2580",
          "2581",
          "2582",
          "2583",
          "2584"
        ]
      }
    },
    {
      "code": "Hushallstyp",
      "selection": {
        "filter": "item",
        "values": [
          "E90"
        ]
      }
    },
    {
      "code": "Alder",
      "selection": {
        "filter": "item",
        "values": [
          "18+"
        ]
      }
    },
    {
      "code": "ContentsCode",
      "selection": {
        "filter": "item",
        "values": [
          "000006SY"
        ]
      }
    }
  ],
  "response": {
    "format": "json"
  }
}
response = requests.post(url, json=payload)
response.raise_for_status()
data = response.json()

In [ ]:
# Unpack the response. SCB returns the year inside the ContentsCode slot,
# which we rename to `Year` for consistency with the other panels.
rows = []

for item in data["data"]:
    row = {
        "Region": item["key"][0],
        "HouseholdType": item["key"][1],
        "Age": item["key"][2],
        "ContentsCode": item["key"][3],
        "Value": item["values"][0]
    }
    rows.append(row)

df_income = pd.DataFrame(rows)
df_income = df_income.rename(columns={
  "ContentsCode" : "Year",
  "Value": "median_household_income"
})
df_income["median_household_income"] = pd.to_numeric(df_income["median_household_income"], errors='coerce')
df_income["Year"] = df_income["Year"].astype(int)


In [ ]:
# Sort within each region and compute the year-on-year growth rate.
df_income = df_income.sort_values(by=["Region", "Year"])
df_income["median_household_income_growth"] = df_income.groupby("Region")["median_household_income"].pct_change()


### 2.3 Urbanization degree

SCB table `MI0810A/TatortGrad`. The share of municipal population living
in localities (tätorter). Only available for a small set of years; values
are forward-filled across years later when building the panel.


In [ ]:
# Endpoint: SCB MI0810A/TatortGrad — urbanization degree (locality share).
url = "https://api.scb.se/OV0104/v1/doris/en/ssd/START/MI/MI0810/MI0810A/TatortGrad"
payload = {
  "query": [
    {
      "code": "Region",
      "selection": {
        "filter": "vs:RegionKommun07EjAggr",
        "values": [
          "0114",
          "0115",
          "0117",
          "0120",
          "0123",
          "0125",
          "0126",
          "0127",
          "0128",
          "0136",
          "0138",
          "0139",
          "0140",
          "0160",
          "0162",
          "0163",
          "0180",
          "0181",
          "0182",
          "0183",
          "0184",
          "0186",
          "0187",
          "0188",
          "0191",
          "0192",
          "0305",
          "0319",
          "0330",
          "0331",
          "0360",
          "0380",
          "0381",
          "0382",
          "0428",
          "0461",
          "0480",
          "0481",
          "0482",
          "0483",
          "0484",
          "0486",
          "0488",
          "0509",
          "0512",
          "0513",
          "0560",
          "0561",
          "0562",
          "0563",
          "0580",
          "0581",
          "0582",
          "0583",
          "0584",
          "0586",
          "0604",
          "0617",
          "0642",
          "0643",
          "0662",
          "0665",
          "0680",
          "0682",
          "0683",
          "0684",
          "0685",
          "0686",
          "0687",
          "0760",
          "0761",
          "0763",
          "0764",
          "0765",
          "0767",
          "0780",
          "0781",
          "0821",
          "0834",
          "0840",
          "0860",
          "0861",
          "0862",
          "0880",
          "0881",
          "0882",
          "0883",
          "0884",
          "0885",
          "0980",
          "1060",
          "1080",
          "1081",
          "1082",
          "1083",
          "1214",
          "1230",
          "1231",
          "1233",
          "1256",
          "1257",
          "1260",
          "1261",
          "1262",
          "1263",
          "1264",
          "1265",
          "1266",
          "1267",
          "1270",
          "1272",
          "1273",
          "1275",
          "1276",
          "1277",
          "1278",
          "1280",
          "1281",
          "1282",
          "1283",
          "1284",
          "1285",
          "1286",
          "1287",
          "1290",
          "1291",
          "1292",
          "1293",
          "1315",
          "1380",
          "1381",
          "1382",
          "1383",
          "1384",
          "1401",
          "1402",
          "1407",
          "1415",
          "1419",
          "1421",
          "1427",
          "1430",
          "1435",
          "1438",
          "1439",
          "1440",
          "1441",
          "1442",
          "1443",
          "1444",
          "1445",
          "1446",
          "1447",
          "1452",
          "1460",
          "1461",
          "1462",
          "1463",
          "1465",
          "1466",
          "1470",
          "1471",
          "1472",
          "1473",
          "1480",
          "1481",
          "1482",
          "1484",
          "1485",
          "1486",
          "1487",
          "1488",
          "1489",
          "1490",
          "1491",
          "1492",
          "1493",
          "1494",
          "1495",
          "1496",
          "1497",
          "1498",
          "1499",
          "1715",
          "1730",
          "1737",
          "1760",
          "1761",
          "1762",
          "1763",
          "1764",
          "1765",
          "1766",
          "1780",
          "1781",
          "1782",
          "1783",
          "1784",
          "1785",
          "1814",
          "1860",
          "1861",
          "1862",
          "1863",
          "1864",
          "1880",
          "1881",
          "1882",
          "1883",
          "1884",
          "1885",
          "1904",
          "1907",
          "1960",
          "1961",
          "1962",
          "1980",
          "1981",
          "1982",
          "1983",
          "1984",
          "2021",
          "2023",
          "2026",
          "2029",
          "2031",
          "2034",
          "2039",
          "2061",
          "2062",
          "2080",
          "2081",
          "2082",
          "2083",
          "2084",
          "2085",
          "2101",
          "2104",
          "2121",
          "2132",
          "2161",
          "2180",
          "2181",
          "2182",
          "2183",
          "2184",
          "2260",
          "2262",
          "2280",
          "2281",
          "2282",
          "2283",
          "2284",
          "2303",
          "2305",
          "2309",
          "2313",
          "2321",
          "2326",
          "2361",
          "2380",
          "2401",
          "2403",
          "2404",
          "2409",
          "2417",
          "2418",
          "2421",
          "2422",
          "2425",
          "2460",
          "2462",
          "2463",
          "2480",
          "2481",
          "2482",
          "2505",
          "2506",
          "2510",
          "2513",
          "2514",
          "2518",
          "2521",
          "2523",
          "2560",
          "2580",
          "2581",
          "2582",
          "2583",
          "2584"
        ]
      }
    },
    {
      "code": "ContentsCode",
      "selection": {
        "filter": "item",
        "values": [
          "MI0810AP"
        ]
      }
    }
  ],
  "response": {
    "format": "json"
  }
}
response = requests.post(url, json=payload)
response.raise_for_status()
data = response.json()

In [ ]:
# Unpack the response. SCB stores the year inside the ContentsCode slot.
rows = []

for item in data["data"]:
    row = {
        "Region": item["key"][0],
        "ContentsCode": item["key"][1],
        "Value": item["values"][0]
    }
    rows.append(row)

df_urbanization = pd.DataFrame(rows)
df_urbanization = df_urbanization.rename(columns={
  "ContentsCode" : "Year",
  "Value": "deg_urbanization"
})
df_urbanization["deg_urbanization"] = df_urbanization["deg_urbanization"].astype(int)
df_urbanization["Year"] = df_urbanization["Year"].astype(int)


### 2.4 Household count

SCB table `BE0101S/HushallT03`. Total number of households per
municipality (all household sizes summed via `TOTAL`). Growth rate is the
per-region year-on-year percentage change.


In [ ]:
# Endpoint: SCB BE0101S/HushallT03 — number of households by household
# size and year. Filtered to the TOTAL (all sizes) line.
url = "https://api.scb.se/OV0104/v1/doris/en/ssd/START/BE/BE0101/BE0101S/HushallT03"
payload = {
  "query": [
    {
      "code": "Region",
      "selection": {
        "filter": "vs:RegionKommun07EjAggr",
        "values": [
          "0114",
          "0115",
          "0117",
          "0120",
          "0123",
          "0125",
          "0126",
          "0127",
          "0128",
          "0136",
          "0138",
          "0139",
          "0140",
          "0160",
          "0162",
          "0163",
          "0180",
          "0181",
          "0182",
          "0183",
          "0184",
          "0186",
          "0187",
          "0188",
          "0191",
          "0192",
          "0305",
          "0319",
          "0330",
          "0331",
          "0360",
          "0380",
          "0381",
          "0382",
          "0428",
          "0461",
          "0480",
          "0481",
          "0482",
          "0483",
          "0484",
          "0486",
          "0488",
          "0509",
          "0512",
          "0513",
          "0560",
          "0561",
          "0562",
          "0563",
          "0580",
          "0581",
          "0582",
          "0583",
          "0584",
          "0586",
          "0604",
          "0617",
          "0642",
          "0643",
          "0662",
          "0665",
          "0680",
          "0682",
          "0683",
          "0684",
          "0685",
          "0686",
          "0687",
          "0760",
          "0761",
          "0763",
          "0764",
          "0765",
          "0767",
          "0780",
          "0781",
          "0821",
          "0834",
          "0840",
          "0860",
          "0861",
          "0862",
          "0880",
          "0881",
          "0882",
          "0883",
          "0884",
          "0885",
          "0980",
          "1060",
          "1080",
          "1081",
          "1082",
          "1083",
          "1214",
          "1230",
          "1231",
          "1233",
          "1256",
          "1257",
          "1260",
          "1261",
          "1262",
          "1263",
          "1264",
          "1265",
          "1266",
          "1267",
          "1270",
          "1272",
          "1273",
          "1275",
          "1276",
          "1277",
          "1278",
          "1280",
          "1281",
          "1282",
          "1283",
          "1284",
          "1285",
          "1286",
          "1287",
          "1290",
          "1291",
          "1292",
          "1293",
          "1315",
          "1380",
          "1381",
          "1382",
          "1383",
          "1384",
          "1401",
          "1402",
          "1407",
          "1415",
          "1419",
          "1421",
          "1427",
          "1430",
          "1435",
          "1438",
          "1439",
          "1440",
          "1441",
          "1442",
          "1443",
          "1444",
          "1445",
          "1446",
          "1447",
          "1452",
          "1460",
          "1461",
          "1462",
          "1463",
          "1465",
          "1466",
          "1470",
          "1471",
          "1472",
          "1473",
          "1480",
          "1481",
          "1482",
          "1484",
          "1485",
          "1486",
          "1487",
          "1488",
          "1489",
          "1490",
          "1491",
          "1492",
          "1493",
          "1494",
          "1495",
          "1496",
          "1497",
          "1498",
          "1499",
          "1715",
          "1730",
          "1737",
          "1760",
          "1761",
          "1762",
          "1763",
          "1764",
          "1765",
          "1766",
          "1780",
          "1781",
          "1782",
          "1783",
          "1784",
          "1785",
          "1814",
          "1860",
          "1861",
          "1862",
          "1863",
          "1864",
          "1880",
          "1881",
          "1882",
          "1883",
          "1884",
          "1885",
          "1904",
          "1907",
          "1960",
          "1961",
          "1962",
          "1980",
          "1981",
          "1982",
          "1983",
          "1984",
          "2021",
          "2023",
          "2026",
          "2029",
          "2031",
          "2034",
          "2039",
          "2061",
          "2062",
          "2080",
          "2081",
          "2082",
          "2083",
          "2084",
          "2085",
          "2101",
          "2104",
          "2121",
          "2132",
          "2161",
          "2180",
          "2181",
          "2182",
          "2183",
          "2184",
          "2260",
          "2262",
          "2280",
          "2281",
          "2282",
          "2283",
          "2284",
          "2303",
          "2305",
          "2309",
          "2313",
          "2321",
          "2326",
          "2361",
          "2380",
          "2401",
          "2403",
          "2404",
          "2409",
          "2417",
          "2418",
          "2421",
          "2422",
          "2425",
          "2460",
          "2462",
          "2463",
          "2480",
          "2481",
          "2482",
          "2505",
          "2506",
          "2510",
          "2513",
          "2514",
          "2518",
          "2521",
          "2523",
          "2560",
          "2580",
          "2581",
          "2582",
          "2583",
          "2584"
        ]
      }
    },
    {
      "code": "Hushallsstorlek",
      "selection": {
        "filter": "item",
        "values": [
          "TOTAL"
        ]
      }
    },
    {
      "code": "ContentsCode",
      "selection": {
        "filter": "item",
        "values": [
          "BE0101CZ"
        ]
      }
    }
  ],
  "response": {
    "format": "json"
  }
}
response = requests.post(url, json=payload)
response.raise_for_status()
data = response.json()

In [ ]:
# Unpack the response, cast to numeric, and compute year-on-year growth
# per region.
rows = []

for item in data["data"]:
    row = {
        "Region": item["key"][0],
        "HouseholdSize": item["key"][1],
        "ContentsCode": item["key"][2],
        "Value": item["values"][0]
    }
    rows.append(row)

df_household = pd.DataFrame(rows)

df_household = df_household.rename(columns={
  "ContentsCode" : "Year",
  "Value": "household_num"
})
df_household
df_household["Year"] = df_household["Year"].astype(int)
df_household["household_num"] = pd.to_numeric(df_household["household_num"], errors='coerce')


df_household = df_household.sort_values(by=["Region", "Year"])
df_household["household_num_growth"] = df_household.groupby("Region")["household_num"].pct_change()

### 2.5 Population density

SCB table `BE0101C/BefArealTathetKon`. Persons per km². The API returns
values for Sex = 1 (men), 2 (women), and 1+2 (both); only the 1+2 row
is kept.


In [ ]:
# Endpoint: SCB BE0101C/BefArealTathetKon — population density by sex.
url = "https://api.scb.se/OV0104/v1/doris/en/ssd/START/BE/BE0101/BE0101C/BefArealTathetKon"
payload = {
  "query": [
    {
      "code": "Region",
      "selection": {
        "filter": "vs:RegionKommun07EjAggr",
        "values": [
          "0114",
          "0115",
          "0117",
          "0120",
          "0123",
          "0125",
          "0126",
          "0127",
          "0128",
          "0136",
          "0138",
          "0139",
          "0140",
          "0160",
          "0162",
          "0163",
          "0180",
          "0181",
          "0182",
          "0183",
          "0184",
          "0186",
          "0187",
          "0188",
          "0191",
          "0192",
          "0305",
          "0319",
          "0330",
          "0331",
          "0360",
          "0380",
          "0381",
          "0382",
          "0428",
          "0461",
          "0480",
          "0481",
          "0482",
          "0483",
          "0484",
          "0486",
          "0488",
          "0509",
          "0512",
          "0513",
          "0560",
          "0561",
          "0562",
          "0563",
          "0580",
          "0581",
          "0582",
          "0583",
          "0584",
          "0586",
          "0604",
          "0617",
          "0642",
          "0643",
          "0662",
          "0665",
          "0680",
          "0682",
          "0683",
          "0684",
          "0685",
          "0686",
          "0687",
          "0760",
          "0761",
          "0763",
          "0764",
          "0765",
          "0767",
          "0780",
          "0781",
          "0821",
          "0834",
          "0840",
          "0860",
          "0861",
          "0862",
          "0880",
          "0881",
          "0882",
          "0883",
          "0884",
          "0885",
          "0980",
          "1060",
          "1080",
          "1081",
          "1082",
          "1083",
          "1214",
          "1230",
          "1231",
          "1233",
          "1256",
          "1257",
          "1260",
          "1261",
          "1262",
          "1263",
          "1264",
          "1265",
          "1266",
          "1267",
          "1270",
          "1272",
          "1273",
          "1275",
          "1276",
          "1277",
          "1278",
          "1280",
          "1281",
          "1282",
          "1283",
          "1284",
          "1285",
          "1286",
          "1287",
          "1290",
          "1291",
          "1292",
          "1293",
          "1315",
          "1380",
          "1381",
          "1382",
          "1383",
          "1384",
          "1401",
          "1402",
          "1407",
          "1415",
          "1419",
          "1421",
          "1427",
          "1430",
          "1435",
          "1438",
          "1439",
          "1440",
          "1441",
          "1442",
          "1443",
          "1444",
          "1445",
          "1446",
          "1447",
          "1452",
          "1460",
          "1461",
          "1462",
          "1463",
          "1465",
          "1466",
          "1470",
          "1471",
          "1472",
          "1473",
          "1480",
          "1481",
          "1482",
          "1484",
          "1485",
          "1486",
          "1487",
          "1488",
          "1489",
          "1490",
          "1491",
          "1492",
          "1493",
          "1494",
          "1495",
          "1496",
          "1497",
          "1498",
          "1499",
          "1715",
          "1730",
          "1737",
          "1760",
          "1761",
          "1762",
          "1763",
          "1764",
          "1765",
          "1766",
          "1780",
          "1781",
          "1782",
          "1783",
          "1784",
          "1785",
          "1814",
          "1860",
          "1861",
          "1862",
          "1863",
          "1864",
          "1880",
          "1881",
          "1882",
          "1883",
          "1884",
          "1885",
          "1904",
          "1907",
          "1960",
          "1961",
          "1962",
          "1980",
          "1981",
          "1982",
          "1983",
          "1984",
          "2021",
          "2023",
          "2026",
          "2029",
          "2031",
          "2034",
          "2039",
          "2061",
          "2062",
          "2080",
          "2081",
          "2082",
          "2083",
          "2084",
          "2085",
          "2101",
          "2104",
          "2121",
          "2132",
          "2161",
          "2180",
          "2181",
          "2182",
          "2183",
          "2184",
          "2260",
          "2262",
          "2280",
          "2281",
          "2282",
          "2283",
          "2284",
          "2303",
          "2305",
          "2309",
          "2313",
          "2321",
          "2326",
          "2361",
          "2380",
          "2401",
          "2403",
          "2404",
          "2409",
          "2417",
          "2418",
          "2421",
          "2422",
          "2425",
          "2460",
          "2462",
          "2463",
          "2480",
          "2481",
          "2482",
          "2505",
          "2506",
          "2510",
          "2513",
          "2514",
          "2518",
          "2521",
          "2523",
          "2560",
          "2580",
          "2581",
          "2582",
          "2583",
          "2584"
        ]
      }
    },
    {
      "code": "Kon",
      "selection": {
        "filter": "item",
        "values": [
          "1",
          "2",
          "1+2"
        ]
      }
    },
    {
      "code": "ContentsCode",
      "selection": {
        "filter": "item",
        "values": [
          "BE0101U1"
        ]
      }
    }
  ],
  "response": {
    "format": "json"
  }
}
response = requests.post(url, json=payload)
response.raise_for_status()
data = response.json()

In [ ]:
# Unpack and keep only the both-sexes (1+2) line.
rows = []

for item in data["data"]:
    row = {
        "Region": item["key"][0],
        "Sex": item["key"][1],
        "ContentsCode": item["key"][2],
        "Value": item["values"][0]
    }
    rows.append(row)

df_density = pd.DataFrame(rows)

df_density = df_density.rename(columns={
    "ContentsCode": "Year",
    "Value": "density"
})
df_density = df_density[df_density["Sex"] == "1+2"].copy()

df_density["Year"] = df_density["Year"].astype(int)
df_density["density"] = pd.to_numeric(df_density["density"], errors='coerce')

### 2.6 Electricity price

SCB table `EN0301A/SSDManadElhandelpris`. Monthly electricity prices
(öre/kWh, excl. taxes and fees) for the flexible-price contract type
(`rorligt`) and the one- or two-dwelling household with electric heating
(`Kundkategori = 3`), reported by bidding zone (SE1–SE4).

Steps: fetch monthly series for 2018-01 to 2025-11 → average to annual →
join the bidding-zone values onto each kommun via `Kommun_bidding.csv`.


In [ ]:
# Endpoint: SCB EN0301A/SSDManadElhandelpris — monthly electricity prices
# by contract type, customer category, and bidding zone.
url = "https://api.scb.se/OV0104/v1/doris/sv/ssd/START/EN/EN0301/EN0301A/SSDManadElhandelpris"
payload = {
  "query": [
    {
      "code": "Avtalstyp",
      "selection": {
        "filter": "item",
        "values": [
          "rorligt"
        ]
      }
    },
    {
      "code": "Kundkategori",
      "selection": {
        "filter": "item",
        "values": [
          "3"
        ]
      }
    },
    {
      "code": "Tid",
      "selection": {
        "filter": "item",
        "values": [
          "2018M01",
          "2018M02",
          "2018M03",
          "2018M04",
          "2018M05",
          "2018M06",
          "2018M07",
          "2018M08",
          "2018M09",
          "2018M10",
          "2018M11",
          "2018M12",
          "2019M01",
          "2019M02",
          "2019M03",
          "2019M04",
          "2019M05",
          "2019M06",
          "2019M07",
          "2019M08",
          "2019M09",
          "2019M10",
          "2019M11",
          "2019M12",
          "2020M01",
          "2020M02",
          "2020M03",
          "2020M04",
          "2020M05",
          "2020M06",
          "2020M07",
          "2020M08",
          "2020M09",
          "2020M10",
          "2020M11",
          "2020M12",
          "2021M01",
          "2021M02",
          "2021M03",
          "2021M04",
          "2021M05",
          "2021M06",
          "2021M07",
          "2021M08",
          "2021M09",
          "2021M10",
          "2021M11",
          "2021M12",
          "2022M01",
          "2022M02",
          "2022M03",
          "2022M04",
          "2022M05",
          "2022M06",
          "2022M07",
          "2022M08",
          "2022M09",
          "2022M10",
          "2022M11",
          "2022M12",
          "2023M01",
          "2023M02",
          "2023M03",
          "2023M04",
          "2023M05",
          "2023M06",
          "2023M07",
          "2023M08",
          "2023M09",
          "2023M10",
          "2023M11",
          "2023M12",
          "2024M01",
          "2024M02",
          "2024M03",
          "2024M04",
          "2024M05",
          "2024M06",
          "2024M07",
          "2024M08",
          "2024M09",
          "2024M10",
          "2024M11",
          "2024M12",
          "2025M01",
          "2025M02",
          "2025M03",
          "2025M04",
          "2025M05",
          "2025M06",
          "2025M07",
          "2025M08",
          "2025M09",
          "2025M10",
          "2025M11"
        ]
      }
    }
  ],
  "response": {
    "format": "json"
  }
}
response = requests.post(url, json=payload)
response.raise_for_status()
data = response.json()

In [ ]:
# Unpack the response. The SCB key tuple is (Contract, Zone, Category,
# Month); the single value is the öre/kWh price.
# SCB returns data in 'key' (dimensions) and 'values' (measurement) pairs
rows = []
for entry in data['data']:
    # Combine keys (Region, Contract, Category, Time) and value (Price) into one list
    row = entry['key'] + entry['values']
    rows.append(row)

# 3. Create DataFrame
# Standard columns for this table: Electricity Area, Contract Type, Category, Time, Price
columns = ["ContractType", "Zone", "Category", "Month", "Price_öre_kWh"]
df_electricity_price = pd.DataFrame(rows, columns=columns)

# 4. Data Cleaning
# Convert price to numeric (float) for statistical analysis
df_electricity_price['Price_öre_kWh'] = pd.to_numeric(df_electricity_price['Price_öre_kWh'], errors='coerce')

# Sort by Region and Month for a clean time-series view
df_electricity_price = df_electricity_price.sort_values(['Zone', 'Month']).reset_index(drop=True)

In [ ]:
# Aggregate the monthly series to an annual mean per bidding zone.
# 1. Extract the Year from the 'Month' column (First 4 characters)
df_electricity_price['Year'] = df_electricity_price['Month'].str[:4].astype(int)

# 2. Filter the data for the required period (2019 - 2024)
df_filtered = df_electricity_price[
    (df_electricity_price['Year'] >= 2018) & 
    (df_electricity_price['Year'] <= 2024)
].copy()

# 3. Group by 'Zone' and 'Year' to calculate the mean price
# We use .reset_index() to keep it in a flat table format (useful for thesis charts)
df_annual_avg = df_filtered.groupby(['Zone', 'Year'])['Price_öre_kWh'].mean().reset_index()

# 4. Rename the column for clarity
df_annual_avg = df_annual_avg.rename(columns={'Price_öre_kWh': 'Avg_Price_öre_kWh'})

# Rename specific columns
# Use 'inplace=True' to apply the change directly to the existing dataframe
df_annual_avg.rename(columns={'Zone': 'bidding_zone'}, inplace=True)

# Verify the changes


In [ ]:
# Load the kommun → bidding-zone mapping (one row per kommun, with the
# zone column matching the SCB price table). Cast Kommun-kod to a
# 4-digit zero-padded string so it joins cleanly with the API outputs.
kommun_bidding = pd.read_csv(
    DATA_DIR / "Kommun_bidding.csv",
    dtype={"Kommun-kod": "object"}
)
kommun_bidding['kommun-kod'] = kommun_bidding['kommun-kod'].astype(str).str.zfill(4)


In [ ]:
# Cross the kommun-zone mapping with the annual price series, producing
# one row per (kommun, year).
df_annual_avg["Year"] = pd.to_numeric(df_annual_avg["Year"], errors="coerce")

# Merge: kommun × year (each kommun repeats for each Year in its bidding_zone)
kommun_bidding_panel = kommun_bidding.merge(
    df_annual_avg,
    on="bidding_zone",
    how="left"
    )

In [ ]:
# Rename to match the `Region` key used elsewhere in this notebook.
kommun_bidding_panel.rename(columns={'kommun-kod': 'Region'}, inplace=True)


### 2.7 Merge API panels

Join every API panel on `(Region, Year)` to build `df_panel`, the long
table of control variables covering 290 kommuner × 2018–2024.


In [ ]:
# Left-join the API panels onto df_density to build df_panel — the long
# table of control variables covering 290 kommuner × 2018–2024. All joins
# are on (Region, Year). After merging, deg_urbanization is forward-filled
# within each region to bridge years where the source table has no entry.
dfs_to_merge = {
    "density": df_density,
    "income": df_income,
    "education": df_education,
    "household": df_household,
    "urbanization": df_urbanization,
    "electricity_price": kommun_bidding_panel
}

# Start from df_density (keeps Region, Sex, Year, density)
df_panel = df_density.copy()

# Merge income variables
df_panel = df_panel.merge(
    df_income[[
        "Region",
        "Year",
        "median_household_income",
        "median_household_income_growth"
    ]],
    on=["Region", "Year"],
    how="left"
)

# Merge education variable
df_panel = df_panel.merge(
    df_education[[
        "Region",
        "Year",
        "bachelor_plus_pct"
    ]],
    on=["Region", "Year"],
    how="left"
)

# Merge household variables
df_panel = df_panel.merge(
    df_household[[
        "Region",
        "Year",
        "household_num",
        "household_num_growth"
    ]],
    on=["Region", "Year"],
    how="left"
)

# Merge urbanization variable
df_panel = df_panel.merge(
    df_urbanization[[
        "Region",
        "Year",
        "deg_urbanization"
    ]],
    on=["Region", "Year"],
    how="left"
)
df_panel = df_panel.merge(
    kommun_bidding_panel[
        ["Region", "Year", "Avg_Price_öre_kWh", "bidding_zone", "kommun_name"]
    ],
    on=["Region", "Year"],
    how="left"
)

df_panel = df_panel.sort_values(by=["Region", "Year"])
df_panel["deg_urbanization"] = df_panel.groupby("Region")["deg_urbanization"].ffill()
df_panel = df_panel[df_panel["Year"].between(2018, 2024)]


## 3. Build the complete dataset

Combine the yearly registration tables (`df_2019` … `df_2024`) with
the API panel, attach lagged (`prev_`) fuel-type columns for scrapping
calculations, and concatenate into a single long-format DataFrame.


In [ ]:
# Add `prev_<fuel>` lag columns to every year's table.
#
# For each fuel and each year t, prev_<fuel> = the same kommun's value
# from year t-1. These lags feed the scrapping calculation later:
#     scr_<fuel> = prev_<fuel> + new_<fuel> - <fuel>     (paper Eq. 3.3)
#
# Missing values (e.g. for the earliest year) are filled with 0.
import pandas as pd
import numpy as np

df_year = {
    2018: df_2018,
    2019: df_2019,
    2020: df_2020,
    2021: df_2021,
    2022: df_2022,
    2023: df_2023,
    2024: df_2024,
}

fuel_cols = [
    'Petrol', 'Diesel', 'Electricity',
    'Electric-hybrids', 'Plug-in-hybrids', 'Total'
]

new_cols = ["new_" + col for col in fuel_cols]

fuel_cols = fuel_cols + new_cols

# 1. Clean the fuel/new_fuel columns in every yearly table:
#    dash -> 0, coerce to numeric, fill NaN with 0.
for year, df in df_year.items():
    df_year[year][fuel_cols] = (
        df[fuel_cols]
        .replace("–", 0)
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0)
    )

# 2. For each year 2019..2024, attach the previous year's values as `prev_*`.
for year in range(2019, 2025):
    prev_year = year - 1

    df_prev = (
        df_year[prev_year][['Kommun-kod'] + fuel_cols]
        .copy()
        .rename(columns={c: f"prev_{c}" for c in fuel_cols})
    )

    df_year[year] = df_year[year].merge(
        df_prev,
        on="Kommun-kod",
        how="left"
    )

    prev_cols = [f"prev_{c}" for c in fuel_cols]
    df_year[year][prev_cols] = df_year[year][prev_cols].fillna(0)

    # Add metadata
    df_year[year]["year"] = year
    df_year[year]["region_code"] = df_year[year]["Kommun-kod"].str[:2]


In [ ]:
# Re-publish the updated yearly DataFrames as named variables.
df_2019 = df_year[2019]
df_2020 = df_year[2020]
df_2021 = df_year[2021]
df_2022 = df_year[2022]
df_2023 = df_year[2023]
df_2024 = df_year[2024]


### 3.1 Attach control variables

Left-join the API control panel onto each yearly registration table,
matching `Kommun-kod` against `Region`.


In [ ]:
# For each year, filter df_panel down to that year and left-join it onto
# the yearly registration table. Keys: Kommun-kod (registrations) =
# Region (panel).
df_final = {}

for year, df_current in df_year.items():
    panel_subset = df_panel[df_panel["Year"] == year].copy()
    
    df_merged = df_current.merge(
        panel_subset,
        left_on="Kommun-kod",
        right_on="Region",
        how="left" 
    )
    
    df_final[year] = df_merged


### 3.2 Concatenate years


In [ ]:
# Stack 2019–2024 into a single long-format DataFrame `df`,
# preserving the year as a column.
years = range(2019, 2025)

df_list = []
for year in years:
    df_year = df_final[year].copy()
    df_year['year'] = year
    df_list.append(df_year)

df = pd.concat(df_list, ignore_index=True)


### 3.3 Compute scrapping counts

Empirical scrapping is recovered from the stock identity:
$$\text{scr}_X = \text{prev}_X + \text{new}_X - X$$
Negative values (which can occur from data-revision noise) are clipped to 0.
See paper Eq. 3.3.


In [ ]:
# Lists used downstream: the five modelled fuel types plus Total, and the
# six continuous control variables that feed the regression.
fuel_types = [
    'Petrol',
    'Diesel',
    'Electricity',
    'Electric-hybrids',
    'Plug-in-hybrids',
    'Total'
]

control_vars = [
    'density',
    'median_household_income_growth',
    'bachelor_plus_pct',
    'household_num_growth',
    'deg_urbanization',
    'Avg_Price_öre_kWh'
]

In [ ]:
# scr_<fuel> = prev + new - current; clip to 0 to absorb any negative
# residuals caused by reclassifications in the underlying registry.
for fuel in fuel_types:
    df[f"scr_{fuel}"] = df[f"prev_{fuel}"] + df[f"new_{fuel}"] - df[fuel]
    df[f"scr_{fuel}"] = df[f"scr_{fuel}"].clip(lower=0)


### 3.4 Owner-type variables

Per-year sheets in `df_onwer_type.xlsx` give counts of vehicles owned by
corporations and by self-employed traders. Two shares are computed:

- `corporation_percentage` — corporate share of the total fleet
- `self_trader_share_in_corp` — self-employed traders' share within the
  corporate group


In [ ]:
# Load yearly owner-type sheets, compute the two shares, and left-join
# them onto the main long-format DataFrame `df` on (year, Kommun-kod).
import pandas as pd
import numpy as np

file_path = DATA_DIR / "data_new" / "df_onwer_type.xlsx"

years = [2019, 2020, 2021, 2022, 2023, 2024]

owner_type_dict = {}

# Load Excel file to access sheets efficiently
excel_file = pd.ExcelFile(file_path)

for year in years:
    sheet_name = str(year)
    
    if sheet_name in excel_file.sheet_names:
        temp_owner = pd.read_excel(excel_file, sheet_name=sheet_name)
        
        # Convert necessary columns to numeric
        target_cols = ['total', 'own_corporations', 'own_corporations_of_self_tradors']
        for col in target_cols:
            temp_owner[col] = pd.to_numeric(temp_owner[col], errors='coerce')
        
        temp_owner['year'] = year
        
        # Share of corporate-owned vehicles in the total fleet
        temp_owner['corporation_percentage'] = (temp_owner['own_corporations'] / temp_owner['total']) * 100
        
        # Share of self-employed traders within the corporate group
        temp_owner['self_trader_share_in_corp'] = np.where(
            temp_owner['own_corporations'] > 0, 
            (temp_owner['own_corporations_of_self_tradors'] / temp_owner['own_corporations']) * 100, 
            0
        )
        
        # Standardise Kommun-kod via float -> int -> str to avoid "132.0"
        # becoming "0132.0" under naive zfill.
        temp_owner["Kommun-kod"] = temp_owner["Kommun-kod"].astype(float).astype(int).astype(str).str.zfill(4)
        
        owner_type_dict[year] = temp_owner
        
        print(f"Processed year {year} into owner_type_dict.")
    else:
        print(f"Warning: Sheet {year} not found.")

# Concatenate all years and merge into the main DataFrame.
df_owner_all = pd.concat(owner_type_dict.values(), ignore_index=True)

df["Kommun-kod"] = df["Kommun-kod"].astype(float).astype(int).astype(str).str.zfill(4)

cols_to_merge = ["year", "Kommun-kod", "corporation_percentage", "self_trader_share_in_corp"]

df = pd.merge(
    df, 
    df_owner_all[cols_to_merge], 
    on=["year", "Kommun-kod"], 
    how="left"
)

print("\nMerge complete. Added 'corporation_percentage' and 'self_trader_share_in_corp' to df.")


## 4. Save the final panel

Write to `output/df.csv` using UTF-8 with BOM (so Kommun-kod leading
zeros survive a round trip through Excel).


In [ ]:
# Force Kommun-kod to a clean 4-digit zero-padded string just before saving.
df_to_save = df.copy()

df_to_save["Kommun-kod"] = (
    df_to_save["Kommun-kod"]
    .astype(str)
    .str.strip()
    .str.zfill(4)
)

df_to_save.to_csv(OUTPUT_DIR / "df.csv", index=False, encoding="utf-8-sig")
